# Active Recall

How do I best remember the key aspects of an implementation so that I can reproduce a piece of code from scratch?<br/>
Right now my thoughts are
- Try to remember the tests name and make sure the test name really connects with what were are trying to do. Here, we test ourselves by stubbing
  out all the test by memory writing in pseudo the steps involved in the test implementation. Ensure we relate everything back to the overall goal.
  A useful tactic here is having like guiding questions that help us tie everything together. What, Why, How, Where, When questions.
- Taking at page from how artists generally produce or reproduce a piece of art quickly
    1. Stub out a general outline of a function/method, struct, `impl` block or trait 
    2. Add initial general detail
    3. Add finer details last

Although it might look like we are trying to memorize the code word for word, they key idea here is how best to fill in the blanks when working on an implementation.<br/>
i.e. what are they key details that we all blanking out on, why, and why are the details important?

My assumption is that this will help in structuring our thinking more clearly and make our own implementation straightforwad because we have a framework in place that
helps us layout our implementations, from high level to low level details.

Honestly, the more I think of this the more it turns out that the leetcode style of solving problems is what we are using to think about project implementation.

# 7.0 Reject Invalid Subscriber #2

## 7.7. Sending A Confirmation Email

### 7.7.x Actix

#### 7.7.1. Static Email

##### 7.7.1.0. Oveview

_**Why?**_<br/>
Check that when we get a new subscriber they get an confirmation email.

_**How?**_<br/>
- First add a test that checks a new subscriber is sent an email. We'll use `wiremock::Mock` to mock the email server that mocks sending
  and email and returning a `200 OK` if it receives the send email request.
- Extract the email client into `subscribe` in order to call `email_client.send_emai()`

##### 7.7.1.1. Red Test

_**What?**_
- What is the name of the test?
  - expected - `subscribe_send_email_for_valid_form_data`
  - actual - `subscribe_sends_confirmation_email_for_valid_form_data`
  - missing - _`sends_confirmation`_
- Stub out solution from memory
```Rust
//! tests/api/helpers.rs ✅

use wiremock::MockServer;

use zero2prod::email_client::EmailClient;

pub struct TestApp {
    pub address: String,
    pub db_pool: PgPool,
    pub email_server: MockServer
}

pub async fn spawn_app() -> TestApp {
    // [...]
    let email_server = MockServer::start().await;
    let configuration = {
        let config = get_config().expect("Failed to read configuration files");
        // [...]
        config.email_client.base_url = &email_server.uri();
        config
    };
    TestApp {
        // [...]
        email_server
    }
}
```
```Rust
//! test/api/subscriptions.rs

#[tokio::test]
async fn subscribe_sends_confirmation_email_on_valid_form_data() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo%40gmail.com";

    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .expect(1) // -> Assertion will be done at tne end of the scope
        .mount(&app.email_server)
        .await; ⚠️

    // Act
    app.post_subscriptions(&body.into()).await; ⚠️

    // Assert
    // Mocke assert on drop ⚠️
}
```
- Update markdown with screenshot.

##### 7.7.1.2. Green Test

_**What?**_<br/>
- Capture `email_client` in `subscribe`
- Make `email_client.send_email` call with dummy data

**Expected Implementation**
```Rust
//! src/routes/subscriptions.rs

pub async fn subscribe(
    form: Form<FormData>, // ?? ❌
    db_pool: web::Data<PgPool>,
    email_client: web::Data<EmailClient>,
) -> HttpResponse {

    // [...]
    
    let new_subscriber = match FormData { // ?? ❌
        Ok(new_subscriber) => new_subscriber,
        Err(_) => return HttpResponse::InternalServerError, // ?? ❌
    };

    if insert_subscriber(&db_pool, &new_subscriber)
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish()
    }
    
    if email_client.send_email(
        &new_subscriber.email, // ?? ❌
        "Welcome",
        "Placeholder HtmlBody",
        "Placeholder TextBody",
    )
    .await
    .is_err()
    {
        return HttpResponse::InternalServerError().finish()
    }
    HttpResponse::Ok().finish()
    
}
```

**Actual Implementation: _Misses_**
```Rust
pub async fn subscribe(
    form: web::Form<FormData>,
    db_pool: web::Data<PgPool>,
    email_client: web::Data<PgPool>,
) {
    let new_subscriber = match form.0.try_into() {
        Ok(form) => form,
        Err(_) => HttpResponse::BadRequest().finish(),
    };

    // [...]
    
    if email_client.send_email(
        new_subscriber.email, // Notice how this call consumes `new_subscriber`
        "Welcome",
        "Placeholder HtmlBody",
        "Placeholder TextBody",
    )
    .await
    // [...]
}
```

### 7.7.x Axum